# PromoSense — Phase 5: Baseline Demand & Causal Uplift Modeling (Google Colab)

This notebook provides a cloud-executable companion for the PromoSense Phase 5 modeling workflow.
It uses the exact same project architecture, time-gated splits, and feature matrices as the local pipeline,
allowing seamless execution on Google Colab (CPU or GPU) without code divergence.

## 1. Environment Setup & Dependency Installation
Ensure required dependencies are installed in the Colab runtime.

In [ ]:
# Install project dependencies in Colab
!pip install -q lightgbm pandas numpy pyarrow scikit-learn scipy psutil joblib matplotlib

## 2. Mount Google Drive or Clone Repository
Point to the PromoSense repository root.

In [ ]:
import os
import sys

# Set project root (adjust path if running from Google Drive)
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"Project root added to sys.path: {PROJECT_ROOT}")

## 3. Run Phase 5 Benchmark (Baseline Demand, Propensity, Uplift Meta-Learners)
Executes the exact same experiment runner used locally, with hardware-adaptive configuration.

In [ ]:
from ml_pipeline.src.models.experiment_runner import Phase5ExperimentRunner

# On Colab, adjust n_jobs based on runtime CPU count (typically 2 for standard Colab, up to 8 for High-RAM)
import multiprocessing
colab_cpus = multiprocessing.cpu_count()
print(f"Detected Colab CPU cores: {colab_cpus}")

runner = Phase5ExperimentRunner(n_jobs=min(4, colab_cpus), random_state=42)
results = runner.run_all_experiments()

## 4. Visualize Baseline Demand & Causal Uplift Diagnostics
Plot the Qini curves and evaluate treatment heterogeneity.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Plot validation Qini curves for candidate uplift learners
plt.figure(figsize=(10, 6))
uplift_val = results["causal_uplift_validation"]

for model_name, res in uplift_val.items():
    if "curve_df" in res:
        curve = pd.DataFrame(res["curve_df"])
        plt.plot(curve["population_fraction"], curve["qini_val"], marker="o", label=f"{model_name} (Qini: {res['qini_score']:.0f})")

# Random benchmark
first_curve = pd.DataFrame(list(uplift_val.values())[0]["curve_df"])
plt.plot(first_curve["population_fraction"], first_curve["random_qini"], "k--", label="Random Benchmark")

plt.title("PromoSense Phase 5: Validation Qini Curves (CATE tau(X))")
plt.xlabel("Fraction of Population Targeted (High to Low Lift)")
plt.ylabel("Cumulative Incremental Gain ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()